# Разширени JavaScript Модели



## Съдържание
1. Въведение към Генератори
2. Генератор като Бегач
3. Proxy Обекти
4. Proxy за Отладка
5. Символи
6. Итератори
7. Шаблонни Литерали и Таг Функции


## 1. Въведение

Генераторите са функции, които (обрзно казано) могат да прекратят изпълнението си и да го възобновят по-късно. Те използват синтаксиса `function*` и ключовата дума `yield`. Казваме, че генераторната функция може да върне временен резултат или в този смисъл - множество последователни резултати от изпълнението.

`yield` се превежда буквално като 'добив'. Т.е. добиваме резултат многократно.

**Основни Преимущества:**
- Отложена оценка (генериране на стойности само при поискване)
- По-малък разход на памет (не трябва да се съхранява целия набор)
- Възможност за пораждане на безкрайни редици (като редицата на Фибоначи)

**Основен Пример:**


In [6]:
// Simple generator function
function* simpleGenerator() {
  console.log('First yield');
  yield 1;
  console.log('Second yield');
  yield 2;
  console.log('Third yield');
  yield 3;
  console.log('Done');
}

const gen = simpleGenerator();

console.log(gen.next()); // { value: 1, done: false }
console.log(gen.next()); // { value: 2, done: false }
console.log(gen.next()); // { value: 3, done: false }
console.log(gen.next()); // { value: undefined, done: true }

First yield
{ value: 1, done: false }
Second yield
{ value: 2, done: false }
Third yield
{ value: 3, done: false }
Done
{ value: undefined, done: true }


### Генератор за стойности в даден обхват


In [7]:
// Generator that produces a range of numbers
function* range(start, end, step = 1) {
  for (let i = start; i <= end; i += step) {
    yield i;
  }
}

// Using the generator
console.log('Range 1 to 5:');
for (const num of range(1, 5)) {
  console.log(num);
}

// Convert to array
const arr = [...range(10, 20, 2)];
console.log('\nRange 10 to 20 (step 2):', arr);

Range 1 to 5:
1
2
3
4
5

Range 10 to 20 (step 2): [ 10, 12, 14, 16, 18, 20 ]


### Генератор за Безкрайна Последователност


In [8]:
// Infinite Fibonacci sequence
function* fibonacci() {
  let a = 0, b = 1;
  while (true) {
    yield a;
    [a, b] = [b, a + b];
  }
}

// Take first 10 Fibonacci numbers
const fib = fibonacci();
const first10 = [];
for (let i = 0; i < 10; i++) {
  first10.push(fib.next().value);
}
console.log('First 10 Fibonacci numbers:', first10);

// ID generator
function* idGenerator() {
  let id = 1;
  while (true) {
    yield `ID-${id++}`;
  }
}

const genId = idGenerator();
console.log('\nGenerated IDs:');
console.log(genId.next().value);
console.log(genId.next().value);
console.log(genId.next().value);

First 10 Fibonacci numbers: [
  0, 1,  1,  2,  3,
  5, 8, 13, 21, 34
]

Generated IDs:
ID-1
ID-2
ID-3


### Двупосочна Комуникация с Генераторите


In [1]:
// Generator that receives values
function* twoWayGenerator() {
  const name = yield 'What is your name?';
  const age = yield `Hello ${name}! How old are you?`;
  return `${name} is ${age} years old`;
}

const conversation = twoWayGenerator();

console.log(conversation.next().value);        // What is your name?
console.log(conversation.next('Alice').value); // Hello Alice! How old are you?
console.log(conversation.next(25).value);      // Alice is 25 years old

What is your name?
Hello Alice! How old are you?
Alice is 25 years old
Hello Alice! How old are you?
Alice is 25 years old


## 2. Генератор като двигател на асинхронни операции

Генераторите могат да се използват за изпълнение на асинхронен код (алтернатива на async/await). Този модел показва как генераторите позволяват управление на потока.


In [ ]:
// Simple async task simulator
function asyncTask(value, delay) {
  return new Promise(resolve => {
    setTimeout(() => {
      console.log(`Task completed with value: ${value}`);
      resolve(value);
    }, delay);
  });
}

// Generator runner
function run(generatorFunc) {
  const iterator = generatorFunc();
  
  function handle(result) {
    if (result.done) return Promise.resolve(result.value);
    
    return Promise.resolve(result.value)
      .then(value => handle(iterator.next(value)))
      .catch(err => handle(iterator.throw(err)));
  }
  
  return handle(iterator.next());
}

// Using generator as async runner
function* asyncFlow() {
  console.log('Starting async flow...');
  
  const result1 = yield asyncTask('Step 1', 100);
  console.log('Got:', result1);
  
  const result2 = yield asyncTask('Step 2', 50);
  console.log('Got:', result2);
  
  const result3 = yield asyncTask('Step 3', 75);
  console.log('Got:', result3);
  
  return 'All done!';
}

run(asyncFlow).then(result => console.log('Final:', result));

Starting async flow...
Promise { <pending> }


Task completed with value: Step 1
Got: Step 1
Task completed with value: Step 2
Got: Step 2
Task completed with value: Step 3
Got: Step 3
Final: All done!


### Обработка на Грешки


In [ ]:
// Async task that might fail
function riskyTask(shouldFail, delay = 100) {
  return new Promise((resolve, reject) => {
    setTimeout(() => {
      if (shouldFail) {
        reject(new Error('Task failed!'));
      } else {
        resolve('Success!');
      }
    }, delay);
  });
}

function* errorHandlingFlow() {
  try {
    const result1 = yield riskyTask(false);
    console.log('First task:', result1);
    
    const result2 = yield riskyTask(true);
    console.log('Second task:', result2); // Won't reach here
  } catch (err) {
    console.log('Caught error:', err.message);
    return 'Recovered from error';
  }
}

run(errorHandlingFlow).then(result => console.log('Result:', result));

## 3. Proxy Обекти

Proxy (посредник) е механизъм за създаване ма обвивка около обект със потребителско поведение.

**Общи Случаи на Употреба:**
- Потвърждаване на отганичения
- Контрол на достъпа до свойства
- Стойности по подразбиране
- Прозрачна връзка с външни източници


In [2]:
// Basic proxy example
const target = {
  name: 'Alice',
  age: 25
};

const handler = {
  get(obj, prop) {
    console.log(`Getting property: ${prop}`);
    return prop in obj ? obj[prop] : 'Property not found';
  },
  set(obj, prop, value) {
    console.log(`Setting ${prop} = ${value}`);
    obj[prop] = value;
    return true; // Indicate success
  }
};

const proxy = new Proxy(target, handler);

console.log(proxy.name);        // Triggers get trap
console.log(proxy.email);       // Property not found
proxy.age = 26;                 // Triggers set trap
console.log(proxy.age);

Getting property: name
Alice
Getting property: email
Property not found
Setting age = 26
Getting property: age
26


### Proxy, което потвърждаващо допустимостта на данните 

In [ ]:
// Create a proxy that validates data
function createValidatedUser() {
  const user = {};
  
  return new Proxy(user, {
    set(obj, prop, value) {
      if (prop === 'age') {
        if (typeof value !== 'number' || value < 0 || value > 150) {
          throw new TypeError('Age must be a number between 0 and 150');
        }
      }
      
      if (prop === 'email') {
        if (!value.includes('@')) {
          throw new TypeError('Invalid email format');
        }
      }
      
      obj[prop] = value;
      return true;
    }
  });
}

const validatedUser = createValidatedUser();

validatedUser.name = 'Bob';
validatedUser.age = 30;
validatedUser.email = 'bob@example.com';
console.log('Valid user:', validatedUser);

try {
  validatedUser.age = 200; // Will throw error
} catch (err) {
  console.log('Validation error:', err.message);
}

try {
  validatedUser.email = 'invalid-email'; // Will throw error
} catch (err) {
  console.log('Validation error:', err.message);
}

### Proxy за Стойности по Подразбиране


In [ ]:
// Proxy that returns default values for missing properties
function withDefaults(target, defaults) {
  return new Proxy(target, {
    get(obj, prop) {
      return prop in obj ? obj[prop] : defaults[prop];
    }
  });
}

const config = { host: 'localhost' };
const configWithDefaults = withDefaults(config, {
  host: '0.0.0.0',
  port: 3000,
  ssl: false
});

console.log('Host:', configWithDefaults.host);   // localhost (from config)
console.log('Port:', configWithDefaults.port);   // 3000 (from defaults)
console.log('SSL:', configWithDefaults.ssl);     // false (from defaults)

### Отрицателна Индексация на Масив с Proxy


In [3]:
// Python-style negative indexing for arrays
function createNegativeArray(arr) {
  return new Proxy(arr, {
    get(target, prop) {
      const index = Number(prop);
      if (Number.isInteger(index)) {
        return target[index < 0 ? target.length + index : index];
      }
      return target[prop];
    }
  });
}

const arr = createNegativeArray(['a', 'b', 'c', 'd', 'e']);

console.log('arr[0]:', arr[0]);     // 'a'
console.log('arr[-1]:', arr[-1]);   // 'e'
console.log('arr[-2]:', arr[-2]);   // 'd'
console.log('arr.length:', arr.length); // 5

arr[0]: a
arr[-1]: e
arr[-2]: d
arr.length: 5


## 4. Proxy за Отладка

Проксито са отлични за отладка чрез логване на всички операции на обект.


In [ ]:
// Create a debugging proxy that logs all operations
function createLoggingProxy(target, name = 'Object') {
  return new Proxy(target, {
    get(obj, prop) {
      console.log(`[${name}] GET ${String(prop)} = ${obj[prop]}`);
      return obj[prop];
    },
    set(obj, prop, value) {
      console.log(`[${name}] SET ${String(prop)} = ${value}`);
      obj[prop] = value;
      return true;
    },
    deleteProperty(obj, prop) {
      console.log(`[${name}] DELETE ${String(prop)}`);
      delete obj[prop];
      return true;
    },
    has(obj, prop) {
      const result = prop in obj;
      console.log(`[${name}] HAS ${String(prop)} = ${result}`);
      return result;
    }
  });
}

const user = createLoggingProxy({ name: 'Alice', age: 25 }, 'User');

console.log('\n--- Operations ---');
user.name;              // Logs GET
user.email = 'a@b.com'; // Logs SET
'age' in user;          // Logs HAS
delete user.age;        // Logs DELETE

console.log('\nFinal object:', Object.keys(user));

### Proxy за Мониторинг на Производителност


In [ ]:
// Proxy that measures function execution time
function createPerformanceProxy(target) {
  return new Proxy(target, {
    get(obj, prop) {
      const value = obj[prop];
      
      if (typeof value === 'function') {
        return function(...args) {
          const start = performance.now();
          const result = value.apply(this, args);
          const end = performance.now();
          console.log(`Method '${String(prop)}' took ${(end - start).toFixed(2)}ms`);
          return result;
        };
      }
      
      return value;
    }
  });
}

const calculator = createPerformanceProxy({
  add(a, b) {
    // Simulate some work
    let sum = 0;
    for (let i = 0; i < 1000000; i++) sum += i;
    return a + b;
  },
  multiply(a, b) {
    let product = 1;
    for (let i = 0; i < 500000; i++) product *= 1.0001;
    return a * b;
  }
});

console.log('Result:', calculator.add(5, 3));
console.log('Result:', calculator.multiply(4, 7));

### Proxy за Само-четене


In [ ]:
// Create a read-only version of an object
function readonly(target) {
  return new Proxy(target, {
    set() {
      console.warn('Cannot modify read-only object');
      return false;
    },
    deleteProperty() {
      console.warn('Cannot delete from read-only object');
      return false;
    }
  });
}

const original = { x: 10, y: 20 };
const readOnly = readonly(original);

console.log('Read value:', readOnly.x);  // Works fine
readOnly.x = 100;                        // Warning, no change
console.log('After attempted change:', readOnly.x); // Still 10
delete readOnly.y;                       // Warning
console.log('Keys:', Object.keys(readOnly)); // Still has both keys

## 5. Символи

Символите са уникални, неизменяеми примитивни стойности, често използвани като ключове на собствосни на обекти, за да се избегнат сблъсъци на имената.

**Ключови Характеристики:**
- Всеки Символ е уникален
- Не са преброени по подразбиране
- Не могат да бъдат случайно презаписани
- Полезни за мета-програмиране


In [4]:
// Creating symbols
const sym1 = Symbol();
const sym2 = Symbol('description');
const sym3 = Symbol('description');

console.log(typeof sym1);           // 'symbol'
console.log(sym2 === sym3);         // false (each Symbol is unique)
console.log(sym2.description);      // 'description'

// Using symbols as object keys
const id = Symbol('id');
const user = {
  name: 'Alice',
  [id]: 12345  // Symbol as computed property
};

console.log('\nUser name:', user.name);
console.log('User ID:', user[id]);

// Symbols are not enumerable
console.log('\nObject.keys:', Object.keys(user));              // ['name']
console.log('for...in:', Object.getOwnPropertyNames(user));     // ['name']
console.log('Symbols:', Object.getOwnPropertySymbols(user));    // [Symbol(id)]

7:13 - This comparison appears to be unintentional because the types 'typeof sym2' and 'typeof sym3' have no overlap.


### Глобален Регистър на Символи


In [ ]:
// Global symbols - shared across realms
const globalSym1 = Symbol.for('app.id');
const globalSym2 = Symbol.for('app.id');

console.log(globalSym1 === globalSym2);  // true (same symbol from registry)

// Get the key for a global symbol
console.log(Symbol.keyFor(globalSym1));  // 'app.id'

// Regular symbols are not in the global registry
const regularSym = Symbol('regular');
console.log(Symbol.keyFor(regularSym));  // undefined

### Известни Символи


In [ ]:
// Symbol.toStringTag - customize Object.prototype.toString
class MyClass {
  get [Symbol.toStringTag]() {
    return 'MyCustomClass';
  }
}

const obj = new MyClass();
console.log(Object.prototype.toString.call(obj)); // [object MyCustomClass]

// Symbol.hasInstance - customize instanceof
class MyArray {
  static [Symbol.hasInstance](instance) {
    return Array.isArray(instance);
  }
}

console.log([] instanceof MyArray);     // true
console.log({} instanceof MyArray);     // false

// Symbol.toPrimitive - customize type coercion
const obj2 = {
  [Symbol.toPrimitive](hint) {
    console.log('Hint:', hint);
    if (hint === 'number') return 42;
    if (hint === 'string') return 'Hello';
    return true;
  }
};

console.log(+obj2);        // 42 (hint: 'number')
console.log(`${obj2}`);    // 'Hello' (hint: 'string')
console.log(obj2 + '');    // 'true' (hint: 'default')

### Приватни Собствосни със Символи


In [ ]:
// Using symbols for "private" properties
const _password = Symbol('password');
const _validate = Symbol('validate');

class Account {
  constructor(username, password) {
    this.username = username;
    this[_password] = password;
  }
  
  [_validate](pwd) {
    return pwd === this[_password];
  }
  
  login(pwd) {
    if (this[_validate](pwd)) {
      console.log(`${this.username} logged in successfully`);
      return true;
    }
    console.log('Invalid password');
    return false;
  }
}

const account = new Account('alice', 'secret123');

console.log('Username:', account.username);      // Accessible
console.log('Password:', account.password);      // undefined (symbol not accessible directly)
console.log('Keys:', Object.keys(account));      // ['username']

account.login('wrong');      // Invalid password
account.login('secret123');  // Logged in successfully

## 6. Итератори

Итераторите предоставят стандартния начин за итериране на данни структури. Всеки обект с метод `Symbol.iterator` е итерируем.

**Протокол на Итератор:**
- Обект с метод `next()`
- `next()` връща `{ value, done }`
- Работи с `for...of`, разпиляване, деструктуриране


In [ ]:
// Manual iterator
const rangeIterator = {
  current: 1,
  last: 5,
  
  next() {
    if (this.current <= this.last) {
      return { value: this.current++, done: false };
    }
    return { done: true };
  }
};

console.log(rangeIterator.next()); // { value: 1, done: false }
console.log(rangeIterator.next()); // { value: 2, done: false }
console.log(rangeIterator.next()); // { value: 3, done: false }
console.log(rangeIterator.next()); // { value: 4, done: false }
console.log(rangeIterator.next()); // { value: 5, done: false }
console.log(rangeIterator.next()); // { done: true }

### Правене на Обекти Итеративни


In [ ]:
// Create an iterable object
const range = {
  start: 1,
  end: 5,
  
  [Symbol.iterator]() {
    let current = this.start;
    const last = this.end;
    
    return {
      next() {
        if (current <= last) {
          return { value: current++, done: false };
        }
        return { done: true };
      }
    };
  }
};

// Now we can use for...of
console.log('for...of loop:');
for (const num of range) {
  console.log(num);
}

// Spread operator
console.log('\nSpread:', [...range]);

// Destructuring
const [first, second, ...rest] = range;
console.log('\nDestructured:', { first, second, rest });

### Итератор за Персонализирана Структура на Данни


In [ ]:
// Linked list with iterator
class Node {
  constructor(value, next = null) {
    this.value = value;
    this.next = next;
  }
}

class LinkedList {
  constructor() {
    this.head = null;
  }
  
  add(value) {
    const node = new Node(value);
    if (!this.head) {
      this.head = node;
    } else {
      let current = this.head;
      while (current.next) {
        current = current.next;
      }
      current.next = node;
    }
  }
  
  [Symbol.iterator]() {
    let current = this.head;
    
    return {
      next() {
        if (current) {
          const value = current.value;
          current = current.next;
          return { value, done: false };
        }
        return { done: true };
      }
    };
  }
}

const list = new LinkedList();
list.add('A');
list.add('B');
list.add('C');

console.log('Linked list values:');
for (const value of list) {
  console.log(value);
}

console.log('\nAs array:', [...list]);

### Използване на Генератори за Итератори


In [ ]:
// Generators make creating iterators easier
class SimpleRange {
  constructor(start, end) {
    this.start = start;
    this.end = end;
  }
  
  *[Symbol.iterator]() {
    for (let i = this.start; i <= this.end; i++) {
      yield i;
    }
  }
}

const simpleRange = new SimpleRange(10, 15);

console.log('Generator-based iterator:');
for (const num of simpleRange) {
  console.log(num);
}

// Tree traversal with generator
class TreeNode {
  constructor(value, children = []) {
    this.value = value;
    this.children = children;
  }
  
  *[Symbol.iterator]() {
    yield this.value;
    for (const child of this.children) {
      yield* child;  // Delegate to child's iterator
    }
  }
}

const tree = new TreeNode('root', [
  new TreeNode('child1', [
    new TreeNode('grandchild1'),
    new TreeNode('grandchild2')
  ]),
  new TreeNode('child2')
]);

console.log('\nTree traversal:', [...tree]);

## 7. Шаблонни Литерали и Таг Функции

Шаблонните литерали предоставят интерполация на низове и многоредови низове. Таг функциите позволяват персонализирана обработка на шаблонни литерали.

**Основни Шаблонни Литерали:**


In [5]:
// Basic template literals
const name = 'Alice';
const age = 25;

const greeting = `Hello, ${name}! You are ${age} years old.`;
console.log(greeting);

// Multi-line strings
const multiline = `
  This is a
  multi-line
  string
`;
console.log(multiline);

// Expressions in template literals
const a = 10, b = 20;
console.log(`Sum: ${a + b}, Product: ${a * b}`);

// Nested templates
const classes = ['primary', 'active'];
const html = `<div class="${classes.join(' ')}">${name}</div>`;
console.log(html);

Hello, Alice! You are 25 years old.

  This is a
  multi-line
  string

Sum: 30, Product: 200
<div class="primary active">Alice</div>


### Таг Функции


In [ ]:
// Basic tag function
function highlight(strings, ...values) {
  console.log('Strings:', strings);
  console.log('Values:', values);
  
  return strings.reduce((result, str, i) => {
    return result + str + (values[i] ? `<mark>${values[i]}</mark>` : '');
  }, '');
}

const name = 'Alice';
const age = 25;
const result = highlight`Name: ${name}, Age: ${age}`;
console.log('\nResult:', result);

### Таг за Експортиране на HTML


In [ ]:
// Tag function for HTML escaping
function html(strings, ...values) {
  const escape = (str) => {
    return String(str)
      .replace(/&/g, '&amp;')
      .replace(/</g, '&lt;')
      .replace(/>/g, '&gt;')
      .replace(/"/g, '&quot;')
      .replace(/'/g, '&#39;');
  };
  
  return strings.reduce((result, str, i) => {
    const value = values[i] !== undefined ? escape(values[i]) : '';
    return result + str + value;
  }, '');
}

const userInput = '<script>alert("XSS")</script>';
const safe = html`<div>User input: ${userInput}</div>`;
console.log('Safe HTML:', safe);

### Таг за Конструиране на SQL Заявки


In [ ]:
// Tag function for SQL-like queries (simulation)
function sql(strings, ...values) {
  const query = strings.reduce((result, str, i) => {
    return result + str + (values[i] !== undefined ? `$${i + 1}` : '');
  }, '');
  
  return {
    text: query,
    values: values
  };
}

const userId = 42;
const status = 'active';

const query = sql`
  SELECT * FROM users
  WHERE id = ${userId}
  AND status = ${status}
`;

console.log('Query:', query.text);
console.log('Values:', query.values);

### Таг за Форматиране на Валута


In [ ]:
// Tag function for currency formatting
function currency(strings, ...values) {
  return strings.reduce((result, str, i) => {
    let value = '';
    if (values[i] !== undefined) {
      value = typeof values[i] === 'number'
        ? `$${values[i].toFixed(2)}`
        : values[i];
    }
    return result + str + value;
  }, '');
}

const price = 19.99;
const tax = 1.5;
const total = price + tax;

console.log(currency`Price: ${price}, Tax: ${tax}, Total: ${total}`);

### Сурови Низове


In [ ]:
// Accessing raw strings (unescaped)
function showRaw(strings, ...values) {
  console.log('Cooked strings:', strings);
  console.log('Raw strings:', strings.raw);
  
  return strings.raw.reduce((result, str, i) => {
    return result + str + (values[i] || '');
  }, '');
}

const path = 'C:\\Users\\Alice';
const result = showRaw`Path: ${path}\nNew line`;
console.log('\nResult:', result);

// String.raw built-in tag
const rawPath = String.raw`C:\Users\Alice\Documents`;
console.log('Raw path:', rawPath);

## Резюме

- **Генераторите** позволяват мързелива оценка и пауза-способни функции с `yield`
- **Генератор бегачи** обработват асинхронния поток (образец преди async/await)
- **Проксито** прехващат операциите на обекти за валидиране, логване и персонализиране
- **Символите** създават уникални ключове на собствосни за мета-програмиране
- **Итераторите** предоставят стандартния протокол за преминаване през структури на данни
- **Шаблонните литерали** позволяват интерполация на низове; **таг функциите** ги обработват

Тези разширени модели отключват мощни мета-програмни способности в JavaScript, позволяващи елегантни решения на сложни проблеми.
